# EmbeddingGemma 2 - sim frame vs. text similarity
Kaggle, GPU T4 x2. Loads `google/embeddinggemma-2` (text + vision encoders, audio skipped) and scores sim camera frames against short text descriptions.

T4 has no bfloat16 and the model card forbids float16 (NaNs), so it runs in **float32** on `cuda:0` (~440M params, ~2 GB; the second T4 is not needed). Generated by `make_notebook.py`.

In [ ]:
!pip install -q -U sentence-transformers transformers

In [ ]:
import torch, transformers, sentence_transformers
print('transformers', transformers.__version__, '| sentence-transformers', sentence_transformers.__version__)
print('torch', torch.__version__)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'cuda:{i}', p.name, f'{p.total_memory/2**30:.1f} GB', 'bf16:', torch.cuda.is_bf16_supported())

Optional: if the HF download needs auth, add a Kaggle secret `HF_TOKEN`.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret('HF_TOKEN'))
    print('HF login ok')
except Exception as e:
    print('No HF_TOKEN secret (fine for public models):', type(e).__name__)

## Images
Downloaded from imgbb with `wget`; images with no URL are decoded from the embedded base64.

In [ ]:
import base64, os, subprocess
IMAGES = [
 {
  "name": "wrist_2_jaws_from_2_sides_of_lever_almost_grasp.png",
  "url": "https://i.ibb.co/W4DShyjR/wrist-2-jaws-from-2-sides-of-lever-almost-grasp.png",
  "texts": [
   "robot jaws grasp door handle"
  ],
  "expected": [],
  "b64": ""
 },
 {
  "name": "dark_gray_pillar_blocks_door.png",
  "url": "https://i.ibb.co/V0sQPBXC/aec60258157e.png",
  "texts": [
   "door handle is hidden",
   "door handle is visible"
  ],
  "expected": [
   "door handle is hidden"
  ],
  "b64": ""
 },
 {
  "name": "latched_slip_door_closed.png",
  "url": "https://i.ibb.co/XrZ11SQr/final-head.png",
  "texts": [
   "door is closed",
   "door is open"
  ],
  "expected": [
   "door is closed"
  ],
  "b64": ""
 },
 {
  "name": "door_slight_open.png",
  "url": "https://i.ibb.co/jvMvHxwC/door-slight-open.png",
  "texts": [
   "door is closed",
   "door is slightly open",
   "door is a little open",
   "door is fully open"
  ],
  "expected": [
   "door is slightly open",
   "door is a little open"
  ],
  "b64": ""
 },
 {
  "name": "slip_mid_arc_not_grasping.png",
  "url": "https://i.ibb.co/4RctMj5D/slip-mid-arc-not-grasping.png",
  "texts": [
   "the robot arm grasps the door handle",
   "the robot arm does not grasp the door handle"
  ],
  "expected": [
   "the robot arm does not grasp the door handle"
  ],
  "b64": ""
 }
]

os.makedirs('imgs', exist_ok=True)
for im in IMAGES:
    im['path'] = os.path.join('imgs', im['name'])
    if im['url']:
        subprocess.run(['wget', '-q', '-O', im['path'], im['url']], check=True)
        src = 'wget ' + im['url']
    else:
        open(im['path'], 'wb').write(base64.b64decode(im['b64']))
        src = 'embedded base64'
    print(f"{im['name']}: {os.path.getsize(im['path'])} bytes via {src}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, len(IMAGES), figsize=(4 * len(IMAGES), 4))
for ax, im in zip(axs, IMAGES):
    ax.imshow(Image.open(im['path']).convert('RGB')); ax.set_title(im['name'], fontsize=7); ax.axis('off')
plt.show()

## Shared scoring helpers

In [ ]:
def report(name, texts, expected, scores, fmt="{:+.4f}"):
    """Print texts ranked by score; return PASS bool if `expected` is non-empty, else None."""
    print(f"\n{name}")
    for t, s in sorted(zip(texts, scores), key=lambda x: -x[1]):
        print(f"  {fmt.format(s)}  {t}")
    if not expected:
        return None
    good = max(s for t, s in zip(texts, scores) if t in expected)
    bad = max(s for t, s in zip(texts, scores) if t not in expected)
    ok = good > bad
    print(f"  expected best in {expected} -> {'PASS' if ok else 'FAIL'} (margin {good - bad:+.4f})")
    return ok


def summarize(results):
    scored = {k: v for k, v in results.items() if v is not None}
    print(f"\n[summary] {sum(scored.values())}/{len(scored)} PASS: "
          + ", ".join(f"{k}={'PASS' if v else 'FAIL'}" for k, v in scored.items()))


## Load model

In [ ]:
import time
from sentence_transformers import SentenceTransformer

t0 = time.time()
model = SentenceTransformer(
    'google/embeddinggemma-2',
    device='cuda:0',
    model_kwargs={'torch_dtype': torch.float32},  # no bf16 on T4; fp16 -> NaN
    config_kwargs={'audio_config': None},         # text + image only
)
model.eval()
n = sum(p.numel() for p in model.parameters())
print(f'loaded in {time.time()-t0:.1f}s, {n/1e6:.0f}M params,',
      f'{torch.cuda.memory_allocated(0)/2**30:.2f} GB on cuda:0, prompts: {list(model.prompts)}')

## Image vs. text similarity
Text is the query (`SearchQuery` prompt), the image is the document (no prefix, per the model card). Embeddings are L2-normalized, so similarity = cosine.

In [ ]:
def enc(x, **kw):
    with torch.no_grad():
        e = model.encode(x, normalize_embeddings=True, convert_to_tensor=True, **kw)
    assert not torch.isnan(e).any(), 'NaN embeddings'
    return e

results = {}
for im in IMAGES:
    e_img = enc({'image': Image.open(im['path']).convert('RGB')})
    e_txt = enc(im['texts'], prompt_name='SearchQuery')
    sims = model.similarity(e_txt, e_img.reshape(1, -1)).squeeze(1).tolist()
    results[im['name']] = report(im['name'], im['texts'], im['expected'], sims)
summarize(results)